In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


from catboost import CatBoostClassifier

RANDOM_STATE = 42
TARGET_COL = "flag"


In [22]:
df = pd.read_pickle("../data/processed/dataset_with_target.pkl")

print(df.shape)
print(df[TARGET_COL].value_counts(normalize=True))


(3000000, 168)
flag
0    0.964519
1    0.035481
Name: proportion, dtype: float64


In [23]:
ohe_cols = [c for c in df.columns if c.startswith("enc_loans_credit_type_")]
ohe_cols, len(ohe_cols)

(['enc_loans_credit_type_0',
  'enc_loans_credit_type_1',
  'enc_loans_credit_type_2',
  'enc_loans_credit_type_3',
  'enc_loans_credit_type_4',
  'enc_loans_credit_type_5',
  'enc_loans_credit_type_6',
  'enc_loans_credit_type_7'],
 8)

In [32]:
df_raw = pd.read_parquet("../data/raw/train_data_0.pq")

df_raw["enc_loans_credit_status"].nunique(dropna=False)

7

In [42]:
df_raw["enc_loans_account_holder_type"].nunique(dropna=False)

7

In [43]:
ohe_holder = (
    pd.get_dummies(
        df_raw[["id", "enc_loans_account_holder_type"]],
        columns=["enc_loans_account_holder_type"],
        prefix="enc_loans_account_holder_type"
    )
    .groupby("id", as_index=False)
    .mean()
)

ohe_holder.shape


(250000, 8)

In [44]:
df_holder = df_status.merge(ohe_holder, on="id", how="left")


KeyboardInterrupt: 

In [33]:
ohe_status = (
    pd.get_dummies(
        df_raw[["id", "enc_loans_credit_status"]],
        columns=["enc_loans_credit_status"],
        prefix="enc_loans_credit_status"
    )
    .groupby("id", as_index=False)
    .mean()
)

ohe_status.shape

(250000, 8)

In [34]:
df_status = df.merge(ohe_status, on="id", how="left")

In [35]:
[c for c in df_status.columns if c.startswith("enc_loans_credit_status_")]

['enc_loans_credit_status_0',
 'enc_loans_credit_status_1',
 'enc_loans_credit_status_2',
 'enc_loans_credit_status_3',
 'enc_loans_credit_status_4',
 'enc_loans_credit_status_5',
 'enc_loans_credit_status_6']

In [24]:
df_small, _ = train_test_split(
    df,
    train_size=300_000,        # нужный размер
    stratify=df[TARGET_COL],   # сохраняем баланс
    random_state=RANDOM_STATE
)

df_small = df_small.reset_index(drop=True)

print(df_small.shape)
print(df_small[TARGET_COL].value_counts(normalize=True))



(300000, 168)
flag
0    0.96452
1    0.03548
Name: proportion, dtype: float64


In [37]:
df_small_status = df_status.loc[df_small.index].copy()
df_small_status.shape

(300000, 175)

In [38]:
X = df_small.drop(columns=[TARGET_COL])
y = df_small[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)


In [39]:
model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]

auc_without_status = roc_auc_score(y_test, y_pred)

auc_without_status


0.7188457103442984

In [28]:
ohe_cols = [c for c in df_small.columns if c.startswith("enc_loans_credit_type_")]

X_no_ohe = df_small.drop(columns=[TARGET_COL] + ohe_cols)
y = df_small[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X_no_ohe, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

auc_without_ohe = roc_auc_score(y_test, y_pred)
auc_without_ohe


0.7144639063320886

In [40]:
X = df_small_status.drop(columns=[TARGET_COL])
y = df_small_status[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape


((240000, 174), (60000, 174))

In [41]:
model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
auc_with_status = roc_auc_score(y_test, y_pred)

auc_with_status


0.7411938628448242

In [12]:
sorted(df_small.columns)

['credit_age_ratio_max',
 'credit_age_ratio_mean',
 'credit_overdue_term_ratio_max',
 'credit_overdue_term_ratio_mean',
 'credit_term_ratio_max',
 'credit_term_ratio_mean',
 'enc_loans_account_cur_max_max',
 'enc_loans_account_cur_max_mean',
 'enc_loans_account_cur_mean_max',
 'enc_loans_account_cur_mean_mean',
 'enc_loans_account_holder_type_max_max',
 'enc_loans_account_holder_type_max_mean',
 'enc_loans_account_holder_type_mean_max',
 'enc_loans_account_holder_type_mean_mean',
 'enc_loans_credit_status_max_max',
 'enc_loans_credit_status_max_mean',
 'enc_loans_credit_status_mean_max',
 'enc_loans_credit_status_mean_mean',
 'enc_loans_credit_type_max_max',
 'enc_loans_credit_type_max_mean',
 'enc_loans_credit_type_mean_max',
 'enc_loans_credit_type_mean_mean',
 'fclose_flag_max_max',
 'fclose_flag_max_mean',
 'fclose_flag_mean_max',
 'fclose_flag_mean_mean',
 'flag',
 'has_90plus_overdue_max',
 'has_90plus_overdue_mean',
 'has_any_bad_payment_max',
 'has_any_bad_payment_mean',
 'has_

In [13]:
# резвимся тут
df_fe = df_small.copy()

# считаем количество плохих платежей (реально существующие колонки)
paym_bad_cnt = (
    df_fe[[c for c in df_fe.columns if c.startswith("enc_paym_") and c.endswith("_max")]] >= 3
).sum(axis=1)

# корректная interaction-фича
df_fe["util_mean_x_paym_bad"] = (
    df_fe["pre_util_mean_mean"] * paym_bad_cnt
)




In [14]:
X = df_fe.drop(columns=[TARGET_COL])
y = df_fe[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False
)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

auc, auc - baseline_auc


(0.7213484220532639, 0.0)

In [10]:
results = []

results.append({
    "experiment": "paym_basic",
    "auc": auc,
    "delta": auc - baseline_auc
})

pd.DataFrame(results)


,experiment,auc,delta
0,paym_basic,0.719839,-0.00151
